# 1. Data Ingestion & Structural Inspection
This notebook loads the 8 official Indian tourism datasets from `../data/raw/` and conducts an initial structural audit (shapes, schemas, column dtypes, missing values, and baseline statistics).


In [105]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings for clean tabular viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_theme(style='whitegrid', font_scale=1.0)
print(f'Environment Ready. Pandas version: {pd.__version__}')


Environment Ready. Pandas version: 3.0.3


## 1. Import All 8 Datasets (Updated through 2026)
Loading all official datasets into dedicated Pandas DataFrames.


In [106]:
# 1. District-wise tourist footfall (2022-2026)
df_district_footfall = pd.read_csv('../data/raw/01_district_wise_tourist_footfall_2022_2024.csv')

# 2. NIDHI hospitality establishments registry (audits through 2026)
df_nidhi_hospitality = pd.read_csv('../data/raw/02_nidhi_hospitality_establishments_registry.csv')

# 3. ASI centrally protected monuments inventory (footfalls through 2026)
df_asi_monuments = pd.read_csv('../data/raw/03_asi_protected_monuments_inventory.csv')

# 4. Inbound tourist exit survey microdata (2023-2026)
df_inbound_survey = pd.read_csv('../data/raw/04_inbound_tourist_survey_microdata_2023_2024.csv')

# 5. Airport monthly passenger traffic (2022-2026)
df_airport_traffic = pd.read_csv('../data/raw/05_airport_passenger_traffic_monthly_2022_2024.csv')

# 6. Annual macro overview (2022-2026)
df_macro_overview = pd.read_csv('../data/raw/06_annual_macro_overview_2022_2024.csv')

# 7. Country-wise FTA summary (2022-2026)
df_fta_summary = pd.read_csv('../data/raw/07_country_wise_fta_summary_2022_2024.csv')

# 8. State and UT consolidated summary (2022-2026)
df_state_ut_summary = pd.read_csv('../data/raw/08_state_ut_consolidated_summary_2022_2023.csv')

print('Successfully imported all 8 datasets into DataFrames!')


Successfully imported all 8 datasets into DataFrames!


## 2. Data Inspection & Description (Before Optimization)
Before modifying or optimizing data types, this section inspects:
* Dimensions and missing value ratios across all 8 datasets.
* Exact column names, current data types, null counts, and sample entries.
* Five-point statistical summaries (`.describe()`) of initial numerical attributes.


In [107]:
# 2.1 Overview of Dataset Dimensions and Missingness
raw_datasets = {
    '01. District Footfall (2022-2026)': df_district_footfall,
    '02. NIDHI Hospitality Registry': df_nidhi_hospitality,
    '03. ASI Monuments Inventory': df_asi_monuments,
    '04. Inbound Tourist Survey (2023-2026)': df_inbound_survey,
    '05. Airport Passenger Traffic (2022-2026)': df_airport_traffic,
    '06. Annual Macro Overview (2022-2026)': df_macro_overview,
    '07. Country-wise FTA Summary': df_fta_summary,
    '08. State/UT Summary (2022-2026)': df_state_ut_summary,
}

summary_df = pd.DataFrame([
    {
        'Dataset': name,
        'Total Rows': len(df),
        'Total Columns': len(df.columns),
        'Total Nulls': df.isnull().sum().sum(),
        'Missing %': f'{(df.isnull().sum().sum() / df.size) * 100:.2f}%'
    }
    for name, df in raw_datasets.items()
])

print('--- Dataset Dimensions & Null Summary ---')
display(summary_df.head(8))


--- Dataset Dimensions & Null Summary ---


,Dataset,Total Rows,Total Columns,Total Nulls,Missing %
0,01. District Footfall (2022-2026),2380,11,1372,5.24%
1,02. NIDHI Hospitality Registry,1650,15,1868,7.55%
2,03. ASI Monuments Inventory,1200,16,3057,15.92%
3,04. Inbound Tourist Survey (2023-2026),2500,18,1353,3.01%
4,05. Airport Passenger Traffic (2022-2026),2400,14,2056,6.12%
5,06. Annual Macro Overview (2022-2026),5,9,1,2.22%
6,07. Country-wise FTA Summary,31,13,11,2.73%
7,08. State/UT Summary (2022-2026),36,18,0,0.00%


In [108]:
# 2.2 Detailed Column Schema & Type Inspection Function
def inspect_columns(df, df_name):
    schema_df = pd.DataFrame({
        'Column': df.columns,
        'Current_Dtype': [str(df[col].dtype) for col in df.columns],
        'Null_Count': [df[col].isnull().sum() for col in df.columns],
        'Null_Pct': [f'{(df[col].isnull().sum() / len(df)) * 100:.1f}%' for col in df.columns],
        'Sample_Value': [df[col].dropna().iloc[0] if not df[col].dropna().empty else None for col in df.columns]
    })
    print(f'=== Schema Inspection: {df_name} ===')
    display(schema_df.head(len(df.columns)))

# Inspect primary datasets
inspect_columns(df_district_footfall, 'df_district_footfall')
inspect_columns(df_nidhi_hospitality, 'df_nidhi_hospitality')
inspect_columns(df_inbound_survey, 'df_inbound_survey')
inspect_columns(df_asi_monuments, 'ASI Monuments Inventory	')
inspect_columns(df_airport_traffic, 'Airport Passenger Traffic (2022-2026)')
inspect_columns(df_macro_overview, ' Annual Macro Overview (2022-2026)')
inspect_columns(df_fta_summary, 'Country-wise FTA Summary')
inspect_columns(df_state_ut_summary, 'State/UT Summary (2022-2026)')



=== Schema Inspection: df_district_footfall ===


,Column,Current_Dtype,Null_Count,Null_Pct,Sample_Value
0,District,str,0,0.0%,Agra
1,State_UT,str,0,0.0%,Uttar Pradesh
2,Year,int64,0,0.0%,2022
3,Domestic_Visitors,int64,0,0.0%,2133912
4,Foreign_Visitors,float64,453,19.0%,48112.0
5,Total_Visitors,int64,0,0.0%,2182024
6,Tourism_Category,str,0,0.0%,Business_Urban
7,Registered_Hotels_Count,float64,415,17.4%,401.0
8,Avg_Stay_Duration_Days,float64,210,8.8%,3.6
9,Tourist_Police_Station_Available,str,0,0.0%,Yes


=== Schema Inspection: df_nidhi_hospitality ===


,Column,Current_Dtype,Null_Count,Null_Pct,Sample_Value
0,Establishment_ID,str,0,0.0%,NIDHI-IND-00001
1,Establishment_Name,str,0,0.0%,Heritage Chandigarh Suites
2,Accommodation_Type,str,0,0.0%,Guest House
3,Star_Classification,str,370,22.4%,Unclassified
4,State_UT,str,0,0.0%,Chandigarh
5,District_City,str,0,0.0%,Chandigarh
6,Pincode,float64,70,4.2%,497236.0
7,Total_Rooms,int64,0,0.0%,4
8,Room_Tariff_Min_INR,int64,0,0.0%,2500
9,Room_Tariff_Max_INR,float64,150,9.1%,6500.0


=== Schema Inspection: df_inbound_survey ===


...

,Column,Current_Dtype,Null_Count,Null_Pct,Sample_Value
0,Year,int64,0,0.0%,2022.00
1,Foreign_Tourist_Arrivals_FTA,int64,0,0.0%,6440000.00
2,International_Tourist_Arrivals_ITA,int64,0,0.0%,13921000.00
3,FTA_YoY_Growth_Pct,float64,1,20.0%,47.83
4,Foreign_Exchange_Earnings_INR_Crore,int64,0,0.0%,177703.00
5,FEE_USD_Billion,float64,0,0.0%,21.40
6,Outbound_Indian_Departures,int64,0,0.0%,20900000.00
7,Outbound_Expenditure_USD_Billion,float64,0,0.0%,14.20
8,Tourism_Share_in_National_GDP_Pct,float64,0,0.0%,4.62


=== Schema Inspection: Country-wise FTA Summary ===


,Column,Current_Dtype,Null_Count,Null_Pct,Sample_Value
0,Country_of_Nationality,str,0,0.0%,USA
1,Region,str,0,0.0%,Americas
2,FTA_2022,float64,1,3.2%,931254.0
3,FTA_2023,float64,1,3.2%,1456789.0
4,FTA_2024,float64,1,3.2%,1804586.0
5,Growth_2023_over_2022_Pct,float64,1,3.2%,56.43
6,Growth_2024_over_2023_Pct,float64,1,3.2%,23.87
7,Market_Share_2024_Pct,float64,1,3.2%,18.13
8,FTA_2025,float64,1,3.2%,1953415.0
9,FTA_2026,float64,1,3.2%,2113471.0


=== Schema Inspection: State/UT Summary (2022-2026) ===


,Column,Current_Dtype,Null_Count,Null_Pct,Sample_Value
0,State_UT,str,0,0.0%,Uttar Pradesh
1,Region,str,0,0.0%,North
2,Domestic_Visitors_2022,int64,0,0.0%,411200000
3,Domestic_Visitors_2023,int64,0,0.0%,478500000
4,Domestic_YoY_Growth_Pct,float64,0,0.0%,16.37
5,Foreign_Visitors_2022,int64,0,0.0%,648986
6,Foreign_Visitors_2023,int64,0,0.0%,1601503
7,Foreign_YoY_Growth_Pct,float64,0,0.0%,146.77
8,Total_Visitors_2023,int64,0,0.0%,480101503
9,Foreign_Visitor_Share_2023_Pct,float64,0,0.0%,0.33


In [109]:
# 2.3 Statistical Description of Key Numerical Data (Before Optimization)
print('--- df_district_footfall: Numerical Statistics (.describe()) ---')
display(df_district_footfall.describe().round(2).head())

print('--- df_inbound_survey: Financial Spend & Nights (.describe()) ---')
display(df_inbound_survey[['Total_Nights_Stayed', 'Estimated_Total_Spend_USD', 'Daily_Spend_INR']].describe().round(2).head())

print('--- df_nidhi_hospitality: Room Counts & Tariffs (.describe()) ---')
display(df_nidhi_hospitality[['Total_Rooms', 'Room_Tariff_Min_INR', 'Room_Tariff_Max_INR', 'Audit_Quality_Score']].describe().round(2).head())


--- df_district_footfall: Numerical Statistics (.describe()) ---


,Year,Domestic_Visitors,Foreign_Visitors,Total_Visitors,Registered_Hotels_Count,Avg_Stay_Duration_Days,Reported_Tourist_Complaints
count,2380.00,2380.00,1927.00,2380.00,1965.00,2170.00,2086.00
mean,2024.00,889458.65,24168.10,909026.69,82.89,2.88,5.41
std,1.41,1446197.35,81819.67,1503685.28,97.62,0.93,6.64
min,2022.00,45425.00,158.00,49018.00,12.00,1.10,0.00
25%,2023.00,323263.50,2639.50,326982.00,34.00,2.10,2.00


--- df_inbound_survey: Financial Spend & Nights (.describe()) ---


,Total_Nights_Stayed,Estimated_Total_Spend_USD,Daily_Spend_INR
count,2500.00,2184.00,1995.00
mean,22.24,4407.02,16627.79
std,11.76,3049.11,6658.03
min,3.00,217.00,4994.00
25%,12.00,1933.75,10814.50


--- df_nidhi_hospitality: Room Counts & Tariffs (.describe()) ---


,Total_Rooms,Room_Tariff_Min_INR,Room_Tariff_Max_INR,Audit_Quality_Score
count,1650.00,1650.00,1500.00,740.00
mean,79.47,4151.21,12652.20,78.17
std,96.44,3343.81,11038.28,11.74
min,2.00,800.00,1800.00,58.00
25%,9.00,1500.00,4000.00,67.80
